# Verify audio pipeline

Confirm an extracted `.wav` file is actually 16kHz mono, and that it
flows correctly through the frozen `wav2vec2-large-960h-lv60-self` audio
encoder.

**What "looks right" means:**
- `soundfile.info(...)` reports `samplerate == 16000` and `channels == 1`.
- The wav2vec2 processor/model runs without error on it.
- The output feature tensor has shape `(batch=1, time, hidden=1024)` --
  1024 is `wav2vec2-large`'s hidden size; `time` will be roughly
  `num_audio_samples / 320` (wav2vec2's ~20ms/frame stride at 16kHz), not
  an exact fixed number.
- The output dtype is `float32`.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import soundfile as sf
import torch
from transformers import Wav2Vec2Model, Wav2Vec2Processor

## Config

Point this at one real extracted `.wav` file -- e.g. one of the
`audio_path` values from notebook 01's manifests.

In [ ]:
# TODO: fill in a real path from notebook 01's manifest audio_path column
SAMPLE_WAV_PATH = Path("/scratch/project_2020712/datasets/extracted_audio/REPLACE_ME.wav")

## Confirm the wav is 16kHz mono

In [ ]:
info = sf.info(str(SAMPLE_WAV_PATH))
print(f"samplerate: {info.samplerate}")
print(f"channels:   {info.channels}")
print(f"duration:   {info.frames / info.samplerate:.2f}s")

assert info.samplerate == 16000, f"expected 16000Hz, got {info.samplerate}"
assert info.channels == 1, f"expected mono, got {info.channels} channels"

## Run it through the frozen wav2vec2 encoder

In [ ]:
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-large-960h-lv60-self")
model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-large-960h-lv60-self")
model.eval()

audio, sample_rate = sf.read(str(SAMPLE_WAV_PATH))
inputs = processor(audio, sampling_rate=sample_rate, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

features = outputs.last_hidden_state
print(f"feature tensor shape: {tuple(features.shape)}")
print(f"feature tensor dtype: {features.dtype}")

## TODO checklist

- [ ] `features.shape == (1, time, 1024)`, with `time` roughly
      `len(audio) / 320`.
- [ ] `features.dtype == torch.float32`.
- [ ] No NaN/Inf values in `features` (`torch.isfinite(features).all()`).